In [ ]:
"""
File: Data_Processing.ipynb

Purpose:
    Provides preprocessing utilities to construct SMB factors from a daily portfolio dataset,
    then smooth and resample the results to a lower frequency.

Classes:
    PortfolioPreprocessor:
        Loads raw CSV data, constructs SMB-related series, neutralizes SMB against other factors
        via OLS regression residuals, and exports a smoothed + resampled CSV.
"""
import pandas as pd
import statsmodels.api as sm


class PortfolioPreprocessor:
  """
  Preprocess portfolio factor data and generate resampled outputs.
  """
  def __init__(
      self,
      input_csv="Portfolio_daily.csv",
      output_csv="biweekly_smoothed.csv",
      date_col="date",
      window_days=14,
      resample_rule="2W"
  ):
      self.input_csv = input_csv
      self.output_csv = output_csv
      self.date_col = date_col
      self.window_days = window_days
      self.resample_rule = resample_rule

      # Decile groups for SMB2
      self.small_cols = ["Lo 10", "2-Dec", "3-Dec", "4-Dec", "5-Dec"]
      self.big_cols   = ["6-Dec", "7-Dec", "8-Dec", "9-Dec", "Hi 10"]

      self.df = None

  # SMB construction
  def construct_smb(self):
    """
    Load raw CSV and construct SMB-related series.

    Return:
        pandas.DataFrame
            DataFrame with new columns:
                date_col, SMB_constructed, SMB2, SMB_constructed_nutr
    """
    # Load data
    df = pd.read_csv(self.input_csv)

    # Create date from year
    df["date"] = pd.to_datetime(df["year"], format="%Y%m%d", errors="coerce")

    # Drop invalid dates and sort
    df = df.dropna(subset=["date"]).sort_values("date").reset_index(drop=True)

    # SMB_constructed
    df["SMB_constructed"] = (df["SMB_res"] + df["RMW_res"] + df["CMA_res"]) / 3

    # SMB2 (Decile-based)
    df["SMB2"] = df[self.small_cols].mean(axis=1) - df[self.big_cols].mean(axis=1)

    # Neutralize SMB_constructed on HML, RMW, CMA
    X = sm.add_constant(df[["HML", "RMW", "CMA"]])
    y = df["SMB_constructed"]
    model = sm.OLS(y, X, missing="drop").fit()
    df["SMB_constructed_nutr"] = model.resid

    self.df = df
    return df


  # Smoothing & resampling
  def smooth_and_resample(self):
    """
    Smooth numeric columns with a rolling mean and resample to a lower frequency.

    Return:
        pandas.DataFrame
            Smoothed and resampled DataFrame saved to output_csv.
    """
    if self.df is None:
        raise ValueError("Run construct_smb() before smoothing.")

    df = self.df.copy()

    # Identify numeric columns
    numeric_cols = df.select_dtypes(include=["number"]).columns.tolist()

    # Rolling mean smoothing
    df[numeric_cols] = (
        df[numeric_cols]
        .rolling(window=self.window_days, min_periods=1)
        .mean()
    )

    # Resample (biweekly)
    df_out = (
        df.resample(self.resample_rule, on=self.date_col)[numeric_cols]
        .mean()
        .reset_index()
    )

    # Save
    df_out.to_csv(self.output_csv, index=False)
    return df_out


if __name__ == "__main__":
  processor = PortfolioPreprocessor(
      input_csv="/content/drive/MyDrive/Colab Notebooks/Portfolio_daily.csv",
      output_csv="biweekly_smoothed_data.csv",
      window_days=14,
      resample_rule="2W"
  )

  processor.construct_smb()
  df_biweekly = processor.smooth_and_resample()
  print(df_biweekly.head())




        date     Lo 10     2-Dec     3-Dec     4-Dec     5-Dec     6-Dec  \
0 1963-07-07 -0.142292 -0.027292 -0.106667 -0.085833 -0.072083  0.070417   
1 1963-07-21  0.048083  0.115346 -0.018143  0.046350 -0.018478  0.013523   
2 1963-08-04 -0.077000 -0.112500 -0.171357 -0.125429 -0.136571 -0.207929   
3 1963-08-18  0.035786  0.041214  0.073143  0.052714  0.209143  0.100786   
4 1963-09-01  0.112571  0.126143  0.203714  0.228286  0.248214  0.199857   

      7-Dec     8-Dec     9-Dec  ...   RMW_res   CMA_res   SMB_res  HML_nutr  \
0 -0.022083  0.027083 -0.072083  ... -0.009449 -0.059750 -0.097602 -0.269362   
1  0.027978  0.052856  0.031336  ...  0.000946 -0.085031 -0.013105 -0.163311   
2 -0.145286 -0.182071 -0.090500  ...  0.004952 -0.071465 -0.019858 -0.091544   
3  0.142214  0.146429  0.228000  ...  0.041113 -0.014553 -0.111994  0.092831   
4  0.250500  0.225571  0.261643  ... -0.006316 -0.003975 -0.034374  0.136683   

   RMW_nutr  CMA_nutr  SMB_nutr  SMB_constructed      SMB2  \
